# Week 7b: PyTorch

The same network as notebook 07, now in **PyTorch**. First you'll check that `autograd` computes exactly the gradients you derived by hand. Then you'll use the real tools: `nn.Module`, loss functions, optimizers, the GPU, and regularisation.

Spec: `docs/superpowers/specs/2026-10-23-deep-learning-design.md`

## Setup (given)
The data from notebook 07, plus **the same starting weights** (same seed, same order), so the gradients can be compared number for number.

In [ ]:
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

sys.path.insert(0, '../service')
from clean import clean_description

df = pd.read_csv('../data/fake_transactions.csv')
df['Clean'] = df['Description'].fillna('').str.strip().apply(clean_description)
data = df[df['Clean'] != '']
text_train, text_test, cat_train, cat_test = train_test_split(
    data['Clean'], data['Category'], test_size=0.2, stratify=data['Category'], random_state=42)
vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4)).fit(text_train)
X_train, X_test = vec.transform(text_train).toarray(), vec.transform(text_test).toarray()
LABELS = sorted(data['Category'].unique())
y_train = np.array([LABELS.index(c) for c in cat_train])
y_test = np.array([LABELS.index(c) for c in cat_test])
N_IN, N_HIDDEN, N_OUT = X_train.shape[1], 64, len(LABELS)

# Notebook 07's starting weights: same rng, same order
rng = np.random.default_rng(0)
W1_np = rng.standard_normal((N_IN, N_HIDDEN)) * np.sqrt(2 / N_IN); b1_np = np.zeros(N_HIDDEN)
W2_np = rng.standard_normal((N_HIDDEN, N_OUT)) * np.sqrt(2 / N_HIDDEN); b2_np = np.zeros(N_OUT)
print(torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

---
## A. Tensors and autograd
A tensor is a NumPy array that can remember how it was computed. With `requires_grad=True`, calling `loss.backward()` fills each tensor's `.grad`: backpropagation done for you.

1. Make tensors `W1, b1, W2, b2` from the `*_np` arrays (`torch.tensor(..., requires_grad=True)`; they stay float64)
2. `Xb, yb` = the first 32 rows of `X_train`, `y_train` as tensors
3. `logits = relu(Xb @ W1 + b1) @ W2 + b2`, then `loss = F.cross_entropy(logits, yb)`, then `loss.backward()`
4. Save `torch_grads = {'W1': W1.grad.numpy(), ...}` for all four

In [ ]:
# TODO


This check (given) recomputes the gradients with notebook 07's backprop (written out here in NumPy) and compares them with autograd's.

In [ ]:
def numpy_backprop(X, y):
    Z1 = X @ W1_np + b1_np; A1 = np.maximum(0, Z1); Z2 = A1 @ W2_np + b2_np
    P = np.exp(Z2 - Z2.max(1, keepdims=True)); P /= P.sum(1, keepdims=True)
    dZ2 = (P - np.eye(N_OUT)[y]) / len(y)
    dZ1 = (dZ2 @ W2_np.T) * (Z1 > 0)
    return {'W1': X.T @ dZ1, 'b1': dZ1.sum(0), 'W2': A1.T @ dZ2, 'b2': dZ2.sum(0)}

ours = numpy_backprop(X_train[:32], y_train[:32])
for k in ours:
    diff = np.abs(ours[k] - torch_grads[k]).max()
    print(f'{k}: max difference {diff:.1e}')
    assert diff < 1e-6, k
print('autograd matches your backprop ✓')

---
## B. `nn.Module`
Real code doesn't create weights by hand. Build `model = nn.Sequential(nn.Linear(N_IN, N_HIDDEN), nn.ReLU(), nn.Linear(N_HIDDEN, N_OUT))`, use `loss_fn = nn.CrossEntropyLoss()` and `opt = torch.optim.SGD(model.parameters(), lr=0.5)`, and train 30 epochs with the given `train_epoch` helper. Print the test accuracy.

**Think:** `CrossEntropyLoss` takes the raw `logits`, not softmax probabilities. Why is it better for the loss to apply softmax itself? (Hint: `log(softmax(z))` of a very small probability.)

In [ ]:
# Given helpers (float32 from here on, as real models use)
Xtr, ytr = torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train)
Xte, yte = torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test)

def train_epoch(model, opt, loss_fn, X, y, batch=64):
    """One pass over the data in shuffled mini-batches; returns the mean training loss."""
    model.train()
    total = 0.0
    for idx in torch.randperm(len(X)).split(batch):
        opt.zero_grad()
        loss = loss_fn(model(X[idx]), y[idx])
        loss.backward()
        opt.step()
        total += float(loss) * len(idx)
    return total / len(X)

@torch.no_grad()
def accuracy(model, X, y):
    model.eval()
    return float((model(X).argmax(1) == y).float().mean())

@torch.no_grad()
def val_loss(model, X, y):
    model.eval()
    return float(F.cross_entropy(model(X), y))

torch.manual_seed(0)

In [ ]:
# TODO


**Why logits:** TODO

---
## C. Optimizers: SGD vs Adam
Train two fresh models (call `torch.manual_seed(0)` before each, so they start the same) for 15 epochs: one with `SGD(lr=0.05)`, one with `Adam(lr=0.001)`. Keep each epoch's loss from `train_epoch` and plot both curves on one chart (log scale helps).

**Think:** Adam keeps a running average of each weight's gradients and their size. Why does that help it move faster early on?

In [ ]:
# TODO


**Your answer:** TODO

---
## D. The GPU
`time_epoch(device)` (given) times one training epoch with the model and data on `'cpu'` or `'cuda'`. Call it for both.

In [ ]:
def time_epoch(device, epochs=3):
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(N_IN, N_HIDDEN), nn.ReLU(), nn.Linear(N_HIDDEN, N_OUT)).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=0.5)
    X, y = Xtr.to(device), ytr.to(device)
    train_epoch(model, opt, nn.CrossEntropyLoss(), X, y)          # warm-up
    if device == 'cuda': torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(epochs):
        train_epoch(model, opt, nn.CrossEntropyLoss(), X, y)
    if device == 'cuda': torch.cuda.synchronize()
    return round((time.perf_counter() - start) / epochs, 4)

In [ ]:
# TODO


**Think:** for this tiny network the GPU may be **no faster**, or even slower. Why? (Hint: what does each mini-batch have to do besides maths, and how much maths is in a 64-row batch?) When would the GPU win easily?

**Your answer:** TODO

---
## E. Overfitting and regularisation
Overfitting is easier to see with little data, so train on only 60 rows (`Xtr_small`, given). Write `run(dropout, weight_decay, epochs=100)` that builds `Linear → ReLU → Dropout(dropout) → Linear`, trains with `AdamW(lr=0.01, weight_decay=weight_decay)`, and records the train loss and `val_loss` on the test set each epoch. Compare `run(0.0, 0.0)` with `run(0.5, 0.05)` side by side, and find the epoch where the plain run's validation loss was lowest (`idxmin()`).

In [ ]:
Xtr_small, ytr_small = Xtr[:60], ytr[:60]

In [ ]:
# TODO


**Think:** in the plain run, what happens to the validation loss while the training loss keeps falling? What do dropout and weight decay each do to slow it down? And what is **early stopping**, and which epoch would it have picked here?

**Your answer:** TODO

---
## F. Other architectures (concepts only)
You've built a **fully connected** network: every input connects to every hidden unit. Three other families matter, and interviewers ask about them:

- **CNN (convolutional network):** small filters (e.g. 3×3 pixels) slide across an image, and the *same* filter is reused everywhere. It finds an edge or an eye wherever it is, with far fewer weights than a fully connected layer.
- **RNN (recurrent network):** reads a sequence one step at a time (a letter, a word, a day of spending) and carries a *hidden state* forward, a memory of what it has seen so far.
- **LSTM:** an RNN with *gates* that decide what to keep, forget and output. Plain RNNs forget long-ago steps because the gradient shrinks a little at every step on the way back (the *vanishing gradient*); LSTMs keep a separate memory line that gradients can travel along.
- **Transformer:** no step-by-step reading. *Attention* lets every word look at every other word directly and decide which ones matter, all in parallel. That parallelism is why they train fast on GPUs and replaced RNNs for text. DistilBERT in notebook 09 is one.

Answer in your own words, 2–3 sentences each:

**1. Why does a CNN suit photos of receipts better than the fully connected network you built?**

**Your answer:** TODO

**2. An RNN reads "JOLLIBEE*4821" one character at a time. What does its hidden state hold after the last character, and why would a plain RNN struggle with a very long description?**

**Your answer:** TODO

**3. How does a transformer read the same description differently, and why does that make it faster to train on a GPU?**

**Your answer:** TODO